# Libraries


In [ ]:
import copy
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from scipy.ndimage import binary_erosion, distance_transform_edt

import kagglehub

# Setup


In [ ]:
DEFAULT_EVAL_CONFIG = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "image_height": 224,
    "image_width": 224,
    "num_classes": 3,
    "batch_size": 16,
    "num_workers": 2,
    "image_col": None,
    "mask_col": None,
    "dataset_handle": "avcuongy/bkai-igh-neopolyp-small-sample",
    "normalize_input": True,
    "input_mean": [0.485, 0.456, 0.406],
    "input_std": [0.229, 0.224, 0.225],
    "strict_weight_loading": True,
    "minimum_model_count": 1,
}

In [ ]:
CONFIG = copy.deepcopy(DEFAULT_EVAL_CONFIG)
CONFIG["device"] = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(CONFIG["device"])

print("Device           :", device)
print("Input size       :", f"{CONFIG['image_height']}x{CONFIG['image_width']}")
print("Number of classes:", CONFIG["num_classes"])

## U-Net Checkpoint Utilities


In [ ]:
def load_checkpoint_file(checkpoint_path, map_location="cpu"):
    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Model checkpoint not found: {checkpoint_path}")

    checkpoint = torch.load(
        checkpoint_path,
        map_location=map_location,
        weights_only=False,
    )

    if not isinstance(checkpoint, dict):
        raise TypeError("Checkpoint must be a dictionary or raw state_dict.")

    return checkpoint


def is_raw_state_dict(value):
    return (
        isinstance(value, dict)
        and bool(value)
        and all(torch.is_tensor(item) for item in value.values())
    )


def extract_model_state_dict(checkpoint):
    if is_raw_state_dict(checkpoint):
        return checkpoint

    for key in (
        "model_state_dict",
        "state_dict",
        "model",
        "net",
        "network",
        "weights",
    ):
        candidate = checkpoint.get(key)

        if isinstance(candidate, nn.Module):
            return candidate.state_dict()
        if is_raw_state_dict(candidate):
            return candidate
        if isinstance(candidate, dict):
            for nested_key in ("state_dict", "model_state_dict", "weights"):
                nested = candidate.get(nested_key)
                if is_raw_state_dict(nested):
                    return nested

    raise KeyError("Cannot find model weights in the checkpoint.")


def strip_prefix(state_dict, prefix):
    return {
        key[len(prefix) :] if key.startswith(prefix) else key: value
        for key, value in state_dict.items()
    }


def align_state_dict_keys(state_dict, model):
    model_state = model.state_dict()
    candidates = [state_dict]
    candidate = state_dict

    # Remove chained wrappers such as _orig_mod.module.
    for _ in range(8):
        stripped = False
        for prefix in ("module.", "model.", "net.", "network.", "_orig_mod."):
            if candidate and all(key.startswith(prefix) for key in candidate):
                candidate = strip_prefix(candidate, prefix)
                candidates.append(candidate)
                stripped = True
                break
        if not stripped:
            break

    def match_score(candidate_state):
        return sum(
            key in model_state and model_state[key].shape == value.shape
            for key, value in candidate_state.items()
        )

    return max(candidates, key=match_score)


def validate_checkpoint_config(checkpoint, model_label, config):
    saved_config = checkpoint.get("config")
    if not isinstance(saved_config, dict):
        return

    for key in ("image_height", "image_width", "num_classes"):
        saved_value = saved_config.get(key)
        if saved_value is None:
            continue
        if int(saved_value) != int(config[key]):
            raise ValueError(
                f"{model_label}: checkpoint {key}={saved_value}, "
                f"but evaluation {key}={config[key]}."
            )


def load_weights_into_model(
    model,
    checkpoint,
    model_label,
    strict=True,
):
    saved_model_name = checkpoint.get("model_name")
    current_model_name = model.__class__.__name__

    if saved_model_name and saved_model_name != current_model_name:
        print(
            f"[{model_label}] Warning: checkpoint model name "
            f"is {saved_model_name}, current class is {current_model_name}."
        )

    state_dict = align_state_dict_keys(
        extract_model_state_dict(checkpoint),
        model,
    )
    result = model.load_state_dict(state_dict, strict=bool(strict))

    print(f"[{model_label}] Class          : {current_model_name}")
    print(f"[{model_label}] Strict loading : {bool(strict)}")
    print(f"[{model_label}] Missing keys   : {len(result.missing_keys)}")
    print(f"[{model_label}] Unexpected keys: {len(result.unexpected_keys)}")

    return result

In [ ]:
# Model checkpoints are loaded after the empty U-Net Models cell.

## Test Dataset


In [ ]:
BASE_CLASS_NAMES = {
    0: "background",
    1: "non-neoplastic",
    2: "neoplastic",
}

BASE_CLASS_COLORS = {
    0: (0, 0, 0),
    1: (0, 255, 0),
    2: (255, 0, 0),
}

num_classes = int(CONFIG["num_classes"])
tab20 = plt.get_cmap("tab20", max(num_classes, 3))

CLASS_NAMES = {
    class_id: BASE_CLASS_NAMES.get(class_id, f"class_{class_id}")
    for class_id in range(num_classes)
}

CLASS_COLORS = {
    class_id: BASE_CLASS_COLORS.get(
        class_id,
        tuple(int(channel * 255) for channel in tab20(class_id)[:3]),
    )
    for class_id in range(num_classes)
}

class_names = CLASS_NAMES

In [ ]:
def decode_ground_truth(mask_rgb):
    mask = np.asarray(mask_rgb)

    if mask.ndim == 2:
        unique_values = np.unique(mask)

        if np.isin(unique_values, [0, 1, 2]).all():
            return mask.astype(np.uint8)

        raise ValueError(
            "Grayscale mask must contain only class IDs 0, 1, 2. "
            f"Found: {unique_values[:20]}"
        )

    if mask.ndim != 3 or mask.shape[-1] != 3:
        raise ValueError(f"Expected HxWx3 RGB mask, got shape={mask.shape}")

    colors = np.array(
        [
            [0, 0, 0],
            [0, 255, 0],
            [255, 0, 0],
        ],
        dtype=np.float32,
    )

    distances = np.sum(
        (mask.astype(np.float32)[:, :, None, :] - colors[None, None, :, :]) ** 2,
        axis=-1,
    )

    return np.argmin(distances, axis=-1).astype(np.uint8)


def pick_column(df, explicit_column=None, candidates=None, required=True):
    if explicit_column is not None:
        if explicit_column not in df.columns:
            raise KeyError(
                f"Column '{explicit_column}' not found. "
                f"Available: {df.columns.tolist()}"
            )
        return explicit_column

    normalized = {str(column).strip().lower(): column for column in df.columns}

    for candidate in candidates or []:
        if candidate.lower() in normalized:
            return normalized[candidate.lower()]

    if len(df.columns) == 1:
        return df.columns[0]

    if required:
        raise KeyError(
            "Cannot infer CSV column automatically. "
            f"Available: {df.columns.tolist()}"
        )

    return None


class SegmentationEvaluationDataset(Dataset):
    IMAGE_COLUMN_CANDIDATES = [
        "image",
        "image_name",
        "image_path",
        "filename",
        "file_name",
        "file",
        "id",
    ]
    MASK_COLUMN_CANDIDATES = [
        "mask",
        "mask_name",
        "mask_path",
        "label",
        "label_path",
        "gt",
        "gt_path",
    ]

    def __init__(
        self,
        dataframe,
        image_dir,
        mask_dir,
        transform=None,
        image_col=None,
        mask_col=None,
    ):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.transform = transform
        self.image_col = pick_column(
            self.df,
            explicit_column=image_col,
            candidates=self.IMAGE_COLUMN_CANDIDATES,
            required=True,
        )
        self.mask_col = pick_column(
            self.df,
            explicit_column=mask_col,
            candidates=self.MASK_COLUMN_CANDIDATES,
            required=False,
        )

    @staticmethod
    def resolve_path(root_dir, value):
        path = Path(str(value))
        return path if path.is_absolute() else Path(root_dir) / path

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        image_path = self.resolve_path(self.image_dir, row[self.image_col])
        mask_value = (
            row[self.mask_col] if self.mask_col is not None else row[self.image_col]
        )
        mask_path = self.resolve_path(self.mask_dir, mask_value)

        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        mask_raw = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)

        if image_bgr is None:
            raise FileNotFoundError(f"Cannot read image: {image_path}")
        if mask_raw is None:
            raise FileNotFoundError(f"Cannot read mask: {mask_path}")

        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

        if mask_raw.ndim == 3:
            mask_raw = cv2.cvtColor(mask_raw, cv2.COLOR_BGR2RGB)

        mask = decode_ground_truth(mask_raw)

        if self.transform is not None:
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]

        if not torch.is_tensor(mask):
            mask = torch.from_numpy(mask)

        return {
            "image": image,
            "mask": mask.long(),
            "image_path": str(image_path),
            "mask_path": str(mask_path),
            "image_name": image_path.name,
        }

In [ ]:
transform_steps = [
    A.Resize(
        height=int(CONFIG["image_height"]),
        width=int(CONFIG["image_width"]),
    )
]

if CONFIG["normalize_input"]:
    transform_steps.append(
        A.Normalize(
            mean=tuple(CONFIG["input_mean"]),
            std=tuple(CONFIG["input_std"]),
        )
    )

transform_steps.append(ToTensorV2())
eval_transform = A.Compose(transform_steps)

dataset_root = Path(kagglehub.dataset_download(CONFIG["dataset_handle"]))
DATA_DIR = dataset_root / "data"
TEST_CSV_PATH = DATA_DIR / "sample" / "test.csv"

if not TEST_CSV_PATH.exists():
    raise FileNotFoundError(f"Test CSV not found: {TEST_CSV_PATH}")

test_df = pd.read_csv(TEST_CSV_PATH)

test_dataset = SegmentationEvaluationDataset(
    dataframe=test_df,
    image_dir=DATA_DIR,
    mask_dir=DATA_DIR,
    transform=eval_transform,
    image_col=CONFIG["image_col"],
    mask_col=CONFIG["mask_col"],
)

test_loader = DataLoader(
    test_dataset,
    batch_size=int(CONFIG["batch_size"]),
    shuffle=False,
    num_workers=int(CONFIG["num_workers"]),
    pin_memory=device.type == "cuda",
)

print("Test samples:", len(test_dataset))

## U-Net Models

Import or define one or more U-Net architectures in the empty cell below, then create this registry:

```python
UNET_EXPERIMENTS = {
    "unet_model_1": {
        "model": model_1,
        "checkpoint_path": "/path/to/model_1.pth",
    },
    "unet_model_2": {
        "model": model_2,
        "checkpoint_path": "/path/to/model_2.pth",
    },
}
```


In [ ]:
UNET_EXPERIMENTS = {
    "unet_model_1": {
        "model": model_1,
        "checkpoint_path": "/path/to/model_1.pth",
    },
}

In [ ]:
def prepare_unet_experiments(experiments, config):
    if not isinstance(experiments, dict):
        raise TypeError("UNET_EXPERIMENTS must be a dictionary.")

    minimum_count = int(config["minimum_model_count"])
    if len(experiments) < minimum_count:
        raise ValueError(
            f"UNET_EXPERIMENTS must contain at least {minimum_count} model."
        )

    prepared = {}

    for model_label, specification in experiments.items():
        if not isinstance(model_label, str) or not model_label.strip():
            raise ValueError("Every U-Net model must have a non-empty string label.")

        if isinstance(specification, nn.Module):
            model = specification
            checkpoint_path = None
        elif isinstance(specification, dict):
            model = specification.get("model")
            checkpoint_path = specification.get("checkpoint_path")
        else:
            raise TypeError(
                f"{model_label}: specification must be an nn.Module or dictionary."
            )

        if not isinstance(model, nn.Module):
            raise TypeError(f"{model_label}: 'model' must be a torch.nn.Module.")

        model = model.to("cpu")

        if checkpoint_path is not None:
            checkpoint = load_checkpoint_file(
                checkpoint_path,
                map_location="cpu",
            )
            validate_checkpoint_config(
                checkpoint=checkpoint,
                model_label=model_label,
                config=config,
            )
            load_weights_into_model(
                model=model,
                checkpoint=checkpoint,
                model_label=model_label,
                strict=config["strict_weight_loading"],
            )
        else:
            print(f"[{model_label}] Using weights already present in model.")

        model.eval()
        prepared[model_label] = model

    return prepared


if "UNET_EXPERIMENTS" not in globals():
    raise NameError(
        "Define UNET_EXPERIMENTS with at least one U-Net model "
        "in the empty U-Net Models cell."
    )

UNET_MODELS = prepare_unet_experiments(
    experiments=UNET_EXPERIMENTS,
    config=CONFIG,
)

print("Prepared model count:", len(UNET_MODELS))
print("Prepared U-Net models:", list(UNET_MODELS))

# Quantitative Evaluation


In [ ]:
def extract_unet_logits(model_output, model_label):
    if torch.is_tensor(model_output):
        logits = model_output
    elif isinstance(model_output, (tuple, list)):
        tensors = [item for item in model_output if torch.is_tensor(item)]
        logits = next((item for item in tensors if item.ndim == 4), None)
    elif isinstance(model_output, dict):
        logits = None
        for key in ("logits", "out", "prediction", "pred"):
            value = model_output.get(key)
            if torch.is_tensor(value):
                logits = value
                break
    elif hasattr(model_output, "logits") and torch.is_tensor(model_output.logits):
        logits = model_output.logits
    else:
        logits = None

    if logits is None:
        raise TypeError(
            f"{model_label}: cannot extract a logits tensor from model output."
        )
    if logits.ndim != 4:
        raise ValueError(
            f"{model_label}: expected logits [B, C, H, W], "
            f"got {tuple(logits.shape)}."
        )

    return logits


@torch.inference_mode()
def predict_unet_batch(
    model,
    images,
    model_label,
    num_classes,
    output_size=None,
):
    logits = extract_unet_logits(model(images), model_label)

    if logits.shape[1] != num_classes:
        raise ValueError(
            f"{model_label}: expected {num_classes} output channels, "
            f"got {logits.shape[1]}."
        )

    if output_size is not None and logits.shape[-2:] != tuple(output_size):
        logits = F.interpolate(
            logits,
            size=tuple(output_size),
            mode="bilinear",
            align_corners=False,
        )

    return torch.argmax(logits, dim=1)


def hd95_binary(pred_mask, true_mask):
    pred = np.asarray(pred_mask, dtype=bool)
    true = np.asarray(true_mask, dtype=bool)

    if not pred.any() or not true.any():
        return np.nan

    pred_surface = pred ^ binary_erosion(pred)
    true_surface = true ^ binary_erosion(true)

    if not pred_surface.any():
        pred_surface = pred
    if not true_surface.any():
        true_surface = true

    distance_to_true = distance_transform_edt(~true_surface)
    distance_to_pred = distance_transform_edt(~pred_surface)
    distances = np.concatenate(
        [distance_to_true[pred_surface], distance_to_pred[true_surface]]
    )

    return float(np.percentile(distances, 95))


def safe_divide(numerator, denominator):
    return np.nan if denominator == 0 else float(numerator / denominator)


@torch.inference_mode()
def evaluate_unet_model(
    model,
    model_label,
    loader,
    device,
    num_classes,
    class_names,
):
    confusion = np.zeros((num_classes, num_classes), dtype=np.int64)
    hd95_values = {class_id: [] for class_id in range(num_classes)}
    model = model.to(device)
    model.eval()

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        targets = batch["mask"].to(device, non_blocking=True).long()
        predictions = predict_unet_batch(
            model=model,
            images=images,
            model_label=model_label,
            num_classes=num_classes,
            output_size=targets.shape[-2:],
        )

        targets_np = targets.cpu().numpy()
        predictions_np = predictions.cpu().numpy()

        if targets_np.min() < 0 or targets_np.max() >= num_classes:
            raise ValueError(
                f"Target IDs must be in [0, {num_classes - 1}], "
                f"found [{targets_np.min()}, {targets_np.max()}]."
            )

        encoded = num_classes * targets_np.reshape(-1) + predictions_np.reshape(-1)
        confusion += np.bincount(
            encoded,
            minlength=num_classes**2,
        ).reshape(num_classes, num_classes)

        for sample_index in range(targets_np.shape[0]):
            for class_id in range(num_classes):
                hd95_values[class_id].append(
                    hd95_binary(
                        predictions_np[sample_index] == class_id,
                        targets_np[sample_index] == class_id,
                    )
                )

    rows = []

    for class_id in range(num_classes):
        true_positive = confusion[class_id, class_id]
        false_positive = confusion[:, class_id].sum() - true_positive
        false_negative = confusion[class_id, :].sum() - true_positive
        gt_pixels = confusion[class_id, :].sum()
        pred_pixels = confusion[:, class_id].sum()
        hd_values = np.asarray(hd95_values[class_id], dtype=np.float64)

        rows.append(
            {
                "model": model_label,
                "class_id": class_id,
                "class_name": class_names[class_id],
                "dice": safe_divide(
                    2 * true_positive,
                    2 * true_positive + false_positive + false_negative,
                ),
                "iou": safe_divide(
                    true_positive,
                    true_positive + false_positive + false_negative,
                ),
                "hd95": (
                    float(np.nanmean(hd_values))
                    if np.isfinite(hd_values).any()
                    else np.nan
                ),
                "precision": safe_divide(
                    true_positive,
                    true_positive + false_positive,
                ),
                "recall": safe_divide(
                    true_positive,
                    true_positive + false_negative,
                ),
                "gt_pixels": int(gt_pixels),
                "pred_pixels": int(pred_pixels),
            }
        )

    class_results = pd.DataFrame(rows)
    foreground = class_results[class_results["class_id"] > 0]
    metric_columns = ["dice", "iou", "hd95", "precision", "recall"]

    def macro_row(class_id, class_name, frame):
        row = {
            "model": model_label,
            "class_id": class_id,
            "class_name": class_name,
            "gt_pixels": int(frame["gt_pixels"].sum()),
            "pred_pixels": int(frame["pred_pixels"].sum()),
        }
        row.update(
            {column: frame[column].mean(skipna=True) for column in metric_columns}
        )
        return row

    results = pd.concat(
        [
            class_results,
            pd.DataFrame(
                [
                    macro_row("macro_fg", "macro_foreground_only", foreground),
                    macro_row("macro_all", "macro_all_classes", class_results),
                ]
            ),
        ],
        ignore_index=True,
    )

    return results, confusion

In [ ]:
evaluation_tables = {}
confusion_matrices = {}

for model_label, unet_model in UNET_MODELS.items():
    print(f"Evaluating: {model_label}")
    model_results, model_confusion = evaluate_unet_model(
        model=unet_model,
        model_label=model_label,
        loader=test_loader,
        device=device,
        num_classes=int(CONFIG["num_classes"]),
        class_names=class_names,
    )
    evaluation_tables[model_label] = model_results
    confusion_matrices[model_label] = model_confusion

    unet_model.to("cpu")
    if device.type == "cuda":
        torch.cuda.empty_cache()

evaluation_df = pd.concat(
    evaluation_tables.values(),
    ignore_index=True,
)

model_comparison_df = evaluation_df[
    evaluation_df["class_id"].eq("macro_fg")
].reset_index(drop=True)

model_comparison_df.round(4)

## Confusion Matrices


In [ ]:
def plot_confusion_matrices(
    confusion_matrices,
    class_names,
    normalize=True,
    max_columns=3,
):
    model_count = len(confusion_matrices)
    column_count = min(int(max_columns), model_count)
    row_count = int(np.ceil(model_count / column_count))
    labels = [class_names[index] for index in range(len(class_names))]
    figure, axes = plt.subplots(
        row_count,
        column_count,
        figsize=(6 * column_count, 5 * row_count),
        squeeze=False,
    )
    flat_axes = axes.ravel()
    image = None

    for axis, (model_label, confusion) in zip(
        flat_axes,
        confusion_matrices.items(),
    ):
        values = confusion.astype(np.float64)

        if normalize:
            row_sums = values.sum(axis=1, keepdims=True)
            values = np.divide(
                values,
                row_sums,
                out=np.zeros_like(values),
                where=row_sums != 0,
            )

        image = axis.imshow(
            values,
            cmap="Blues",
            vmin=0.0,
            vmax=1.0 if normalize else None,
        )
        axis.set(
            xticks=np.arange(len(labels)),
            yticks=np.arange(len(labels)),
            xticklabels=labels,
            yticklabels=labels,
            xlabel="Predicted class",
            ylabel="Ground-truth class",
            title=model_label,
        )
        plt.setp(axis.get_xticklabels(), rotation=30, ha="right")
        threshold = values.max() / 2 if values.size else 0

        for row in range(values.shape[0]):
            for column in range(values.shape[1]):
                text = (
                    f"{values[row, column]:.3f}"
                    if normalize
                    else f"{int(values[row, column])}"
                )
                axis.text(
                    column,
                    row,
                    text,
                    ha="center",
                    va="center",
                    color=("white" if values[row, column] > threshold else "black"),
                )

    for unused_axis in flat_axes[model_count:]:
        unused_axis.axis("off")

    if image is not None:
        figure.colorbar(
            image,
            ax=flat_axes[:model_count].tolist(),
            shrink=0.8,
        )

    figure.suptitle(
        "Normalized Pixel Confusion Matrices"
        if normalize
        else "Pixel Confusion Matrices"
    )
    plt.show()


plot_confusion_matrices(
    confusion_matrices=confusion_matrices,
    class_names=class_names,
    normalize=True,
)

# Qualitative Comparison


In [ ]:
def colorize_mask(mask, class_colors):
    color_mask = np.zeros((*mask.shape, 3), dtype=np.uint8)

    for class_id, color in class_colors.items():
        color_mask[mask == class_id] = color

    return color_mask


def denormalize_image(
    image_tensor,
    normalize_input=True,
    mean=(0.485, 0.456, 0.406),
    std=(0.229, 0.224, 0.225),
):
    image = image_tensor.detach().cpu().permute(1, 2, 0).numpy()

    if normalize_input:
        image = image * np.asarray(std, dtype=np.float32)
        image = image + np.asarray(mean, dtype=np.float32)

    return np.clip(image, 0.0, 1.0)


def get_unet_model(models, model_label):
    if model_label not in models:
        raise KeyError(
            f"Unknown model label: {model_label}. " f"Available models: {list(models)}"
        )
    return models[model_label]


def load_visualization_sample(
    image_path,
    transform,
    mask_path=None,
):
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")

    image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    target = None

    if mask_path is not None:
        mask_path = Path(mask_path)
        if not mask_path.exists():
            raise FileNotFoundError(f"Mask not found: {mask_path}")

        mask_raw = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
        if mask_raw is None:
            raise FileNotFoundError(f"Cannot read mask: {mask_path}")
        if mask_raw.ndim == 3:
            mask_raw = cv2.cvtColor(mask_raw, cv2.COLOR_BGR2RGB)

        transformed = transform(
            image=image,
            mask=decode_ground_truth(mask_raw),
        )
        image_tensor = transformed["image"]
        transformed_mask = transformed["mask"]
        target = (
            transformed_mask.cpu().numpy()
            if torch.is_tensor(transformed_mask)
            else np.asarray(transformed_mask)
        )
    else:
        image_tensor = transform(image=image)["image"]

    return image_path, image_tensor, target


@torch.inference_mode()
def visualize_one_model_n_samples(
    model,
    model_label,
    dataset,
    device,
    num_classes,
    class_colors,
    num_samples=3,
    indices=None,
    normalize_input=True,
    input_mean=(0.485, 0.456, 0.406),
    input_std=(0.229, 0.224, 0.225),
):
    if len(dataset) == 0:
        raise ValueError("The evaluation dataset is empty.")
    if int(num_samples) < 1:
        raise ValueError("num_samples must be >= 1.")

    if indices is None:
        sample_count = min(int(num_samples), len(dataset))
        indices = np.linspace(
            0,
            len(dataset) - 1,
            sample_count,
            dtype=int,
        ).tolist()
    else:
        indices = [int(index) for index in indices]

    invalid_indices = [index for index in indices if index < 0 or index >= len(dataset)]
    if invalid_indices:
        raise IndexError(
            f"Dataset indices out of range: {invalid_indices}. "
            f"Valid range: 0..{len(dataset) - 1}."
        )

    model = model.to(device)
    model.eval()
    figure, axes = plt.subplots(
        len(indices),
        3,
        figsize=(13, 4 * len(indices)),
        squeeze=False,
    )

    for row, dataset_index in enumerate(indices):
        sample = dataset[dataset_index]
        image_tensor = sample["image"]
        target = sample["mask"].cpu().numpy()
        prediction = (
            predict_unet_batch(
                model=model,
                images=image_tensor.unsqueeze(0).to(device),
                model_label=model_label,
                num_classes=num_classes,
                output_size=target.shape,
            )[0]
            .cpu()
            .numpy()
        )

        axes[row, 0].imshow(
            denormalize_image(
                image_tensor,
                normalize_input=normalize_input,
                mean=input_mean,
                std=input_std,
            )
        )
        axes[row, 0].set_title(f"Sample {dataset_index}: {sample['image_name']}")
        axes[row, 1].imshow(colorize_mask(target, class_colors))
        axes[row, 1].set_title("Ground truth")
        axes[row, 2].imshow(colorize_mask(prediction, class_colors))
        axes[row, 2].set_title(f"{model_label} prediction")

        for axis in axes[row]:
            axis.axis("off")

    model.to("cpu")
    if device.type == "cuda":
        torch.cuda.empty_cache()

    figure.tight_layout()
    plt.show()


@torch.inference_mode()
def visualize_one_model_image_path(
    model,
    model_label,
    image_path,
    device,
    transform,
    num_classes,
    class_colors,
    mask_path=None,
    normalize_input=True,
    input_mean=(0.485, 0.456, 0.406),
    input_std=(0.229, 0.224, 0.225),
):
    image_path, image_tensor, target = load_visualization_sample(
        image_path=image_path,
        mask_path=mask_path,
        transform=transform,
    )
    output_size = target.shape if target is not None else tuple(image_tensor.shape[-2:])
    model = model.to(device)
    model.eval()
    prediction = (
        predict_unet_batch(
            model=model,
            images=image_tensor.unsqueeze(0).to(device),
            model_label=model_label,
            num_classes=num_classes,
            output_size=output_size,
        )[0]
        .cpu()
        .numpy()
    )
    model.to("cpu")

    if device.type == "cuda":
        torch.cuda.empty_cache()

    column_count = 3 if target is not None else 2
    figure, axes = plt.subplots(
        1,
        column_count,
        figsize=(5 * column_count, 5),
        squeeze=False,
    )
    axes = axes[0]
    axes[0].imshow(
        denormalize_image(
            image_tensor,
            normalize_input=normalize_input,
            mean=input_mean,
            std=input_std,
        )
    )
    axes[0].set_title(image_path.name)

    if target is not None:
        axes[1].imshow(colorize_mask(target, class_colors))
        axes[1].set_title("Ground truth")
        prediction_axis = axes[2]
    else:
        prediction_axis = axes[1]

    prediction_axis.imshow(colorize_mask(prediction, class_colors))
    prediction_axis.set_title(f"{model_label} prediction")

    for axis in axes:
        axis.axis("off")

    figure.tight_layout()
    plt.show()
    return prediction


@torch.inference_mode()
def visualize_multiple_models_image_path(
    models,
    image_path,
    device,
    transform,
    num_classes,
    class_colors,
    mask_path=None,
    normalize_input=True,
    input_mean=(0.485, 0.456, 0.406),
    input_std=(0.229, 0.224, 0.225),
):
    if len(models) <= 2:
        raise ValueError(
            "visualize_multiple_models_image_path requires more than two models."
        )

    image_path, image_tensor, target = load_visualization_sample(
        image_path=image_path,
        mask_path=mask_path,
        transform=transform,
    )
    output_size = target.shape if target is not None else tuple(image_tensor.shape[-2:])
    predictions = {}

    for model_label, model in models.items():
        model = model.to(device)
        model.eval()
        predictions[model_label] = (
            predict_unet_batch(
                model=model,
                images=image_tensor.unsqueeze(0).to(device),
                model_label=model_label,
                num_classes=num_classes,
                output_size=output_size,
            )[0]
            .cpu()
            .numpy()
        )
        model.to("cpu")

        if device.type == "cuda":
            torch.cuda.empty_cache()

    column_count = 3 if target is not None else 2
    figure, axes = plt.subplots(
        len(models),
        column_count,
        figsize=(5 * column_count, 4 * len(models)),
        squeeze=False,
    )
    display_image = denormalize_image(
        image_tensor,
        normalize_input=normalize_input,
        mean=input_mean,
        std=input_std,
    )

    for row, model_label in enumerate(models):
        axes[row, 0].imshow(display_image)
        axes[row, 0].set_title(image_path.name)
        axes[row, 0].set_ylabel(
            model_label,
            rotation=0,
            labelpad=55,
            va="center",
            fontsize=11,
            fontweight="bold",
        )

        if target is not None:
            axes[row, 1].imshow(colorize_mask(target, class_colors))
            axes[row, 1].set_title("Ground truth")
            prediction_axis = axes[row, 2]
        else:
            prediction_axis = axes[row, 1]

        prediction_axis.imshow(colorize_mask(predictions[model_label], class_colors))
        prediction_axis.set_title("Prediction")

        for axis in axes[row]:
            axis.axis("off")

    figure.tight_layout()
    plt.show()
    return predictions

### One Model: Visualize N Test Samples


In [ ]:
SELECTED_MODEL_LABEL = next(iter(UNET_MODELS))
N_SAMPLES = 3

SAMPLE_INDICES = None

visualize_one_model_n_samples(
    model=get_unet_model(UNET_MODELS, SELECTED_MODEL_LABEL),
    model_label=SELECTED_MODEL_LABEL,
    dataset=test_dataset,
    device=device,
    num_classes=int(CONFIG["num_classes"]),
    class_colors=CLASS_COLORS,
    num_samples=N_SAMPLES,
    indices=SAMPLE_INDICES,
    normalize_input=CONFIG["normalize_input"],
    input_mean=CONFIG["input_mean"],
    input_std=CONFIG["input_std"],
)

### One Model: Visualize a Specific Image Path


In [ ]:
SINGLE_MODEL_LABEL = next(iter(UNET_MODELS))
SINGLE_IMAGE_PATH = None
SINGLE_MASK_PATH = None

if SINGLE_IMAGE_PATH is None:
    print("Set SINGLE_IMAGE_PATH to visualize a specific image.")
else:
    single_path_prediction = visualize_one_model_image_path(
        model=get_unet_model(UNET_MODELS, SINGLE_MODEL_LABEL),
        model_label=SINGLE_MODEL_LABEL,
        image_path=SINGLE_IMAGE_PATH,
        mask_path=SINGLE_MASK_PATH,
        device=device,
        transform=eval_transform,
        num_classes=int(CONFIG["num_classes"]),
        class_colors=CLASS_COLORS,
        normalize_input=CONFIG["normalize_input"],
        input_mean=CONFIG["input_mean"],
        input_std=CONFIG["input_std"],
    )

### Multiple Models: Visualize a Specific Image Path


In [ ]:
MULTI_MODEL_IMAGE_PATH = "../data/train/train/1c4aa179344271fa54aa1b27918630bf.jpeg"
MULTI_MODEL_MASK_PATH = (
    "../data/train_gt/train_gt/1c4aa179344271fa54aa1b27918630bf.jpeg"
)

if len(UNET_MODELS) <= 2:
    print(
        "Multiple-model path visualization requires more than two models. "
        f"Current model count: {len(UNET_MODELS)}"
    )
elif MULTI_MODEL_IMAGE_PATH is None:
    print("Set MULTI_MODEL_IMAGE_PATH to compare all models on one image.")
else:
    multi_path_predictions = visualize_multiple_models_image_path(
        models=UNET_MODELS,
        image_path=MULTI_MODEL_IMAGE_PATH,
        mask_path=MULTI_MODEL_MASK_PATH,
        device=device,
        transform=eval_transform,
        num_classes=int(CONFIG["num_classes"]),
        class_colors=CLASS_COLORS,
        normalize_input=CONFIG["normalize_input"],
        input_mean=CONFIG["input_mean"],
        input_std=CONFIG["input_std"],
    )